# 08 · Manuscript numbers and editable tables

Build the paper's numerical cross-checks and its two main tables and thirteen supplementary tables from the results of notebooks 01–06. Inspect each generated table in its cell. The numeric macro file and LaTeX tables are written under `results/manuscript/` for verification. The manuscript source is maintained separately. Figures are produced in notebook 07.

In [1]:
import json
from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
TAB = ROOT / "results" / "tables"
OUT = ROOT / "results" / "manuscript"; (OUT / "tables").mkdir(parents=True, exist_ok=True)
YEARS = np.arange(2008, 2025)
R = lambda name: pd.read_csv(TAB / f"{name}.csv")
macros = {}
def fmt(x, d=2):
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and d == 0):
        return f"{int(round(x)):,}".replace(",", "{,}")
    s = f"{x:.{d}f}"
    return s.replace("-", "$-$") if s.startswith("-") else s
def put(name, value, d=2):
    assert name.isalpha(), name
    if not isinstance(value, str) and not np.isfinite(value):
        value = "--"
    macros[name] = value if isinstance(value, str) else fmt(value, d)

## 1. Sample and data

In [2]:
section_start = len(macros)
meta = json.loads((ROOT / "results" / "01_reconstruction_meta.json").read_text())
flow = R("sample_flow_by_year"); cov = R("annual_coverage"); cn = cov[cov["sample"] == "newspapers"].set_index("year")
fn = flow[(flow["sample"] == "newspapers") & (flow.year != "Total")].copy(); fn["year"] = fn.year.astype(int); fn = fn.set_index("year")
put("NRawRows", meta["flow"]["raw_rows"], 0); put("NDuplicates", meta["flow"]["exact_duplicates_removed"], 0)
put("NBadDates", meta["flow"]["invalid_start_date_removed"], 0)
put("NActions", meta["eligible_newspapers"], 0); put("NAllMedia", meta["eligible_all_media"], 0)
put("NNewsBeforeFilters", int(fn.actions.sum()), 0); put("NNoValid", int(fn.no_valid_tactic.sum()), 0); put("NRepeated", int(fn.repeated_valid_code.sum()), 0)
put("NMulti", int(cn.multi_tactic.sum()), 0); put("ShareMulti", 100 * cn.multi_tactic.sum() / cn.actions.sum(), 0)
put("NPairsObserved", int(cn.pairs_in_actions.sum()), 0)
put("NResidual", int(fn.eligible_with_residual_99.sum()), 0); put("NSixFields", int(fn.eligible_six_fields.sum()), 0)
put("ActionsMin", int(cn.actions.min()), 0); put("ActionsMinYear", str(int(cn.actions.idxmin())))
put("ActionsMax", int(cn.actions.max()), 0); put("ActionsMaxYear", str(int(cn.actions.idxmax())))
put("MeanKFirst", cn.mean_k.loc[2008]); put("MeanKMax", cn.mean_k.max()); put("MeanKMaxYear", str(int(cn.mean_k.idxmax())))
put("MeanKLast", cn.mean_k.loc[2024]); put("ShareMultiFirst", 100 * cn.share_multi.loc[2008], 0); put("ShareMultiMaxv", 100 * cn.share_multi.max(), 0)
put("ShareMultiMaxYear", str(int(cn.share_multi.idxmax())))
put("DaysWithActions", int(cn.days_with_actions.sum()), 0)
marg = R("tactic_counts_by_year").set_index("year"); marg.columns = marg.columns.astype(int)
share = marg.div(cn.actions, axis=0) * 100
put("NTacticsFifty", int((marg.sum() >= 50).sum()), 0)
for code_, nm in [(29, "Looting"), (19, "Occupation"), (30, "Explosives"), (27, "Arson"), (18, "Strike"), (1, "March"), (22, "Roadblock"), (10, "Chants"), (11, "Boycott")]:
    put(f"Share{nm}Min", share[code_].min(), 1); put(f"Share{nm}MinYear", str(int(share[code_].idxmin())))
    put(f"Share{nm}Max", share[code_].max(), 1); put(f"Share{nm}MaxYear", str(int(share[code_].idxmax())))
    put(f"Share{nm}First", share.loc[2008, code_], 1); put(f"Share{nm}Last", share.loc[2024, code_], 1)
tvd = R("tactic_profile_tvd_year_vs_rest").set_index("year").tvd_year_vs_rest
put("TVDMin", tvd.min()); put("TVDMax", tvd.max()); put("TVDMaxYear", str(int(tvd.idxmax())))
new_values = list(macros.items())[section_start:]
display(pd.DataFrame(new_values, columns=["Macro", "Value"]))

,Macro,Value
0,NRawRows,"38{,}551"
1,NDuplicates,24
2,NBadDates,3
3,NActions,"27{,}488"
4,NAllMedia,"34{,}118"
...,...,...
78,ShareBoycottFirst,0.0
79,ShareBoycottLast,0.0
80,TVDMin,0.11
81,TVDMax,0.28


## 2. Reference models and sampler checks

In [3]:
section_start = len(macros)
mcmc = R("mcmc_diagnostics").set_index("spec")
crosscheck_path = ROOT / "results" / "02_kernel_crosscheck.json"
xc = json.loads(crosscheck_path.read_text()) if crosscheck_path.exists() else {}
uni = R("kernel_uniformity_test")
put("NStrataNtwo", int(mcmc.loc["N2", "strata"]), 0); put("NStrataNone", int(mcmc.loc["N1", "strata"]), 0)
put("NFields", int(mcmc.loc["N1", "occupied_fields"]), 0)
put("AcceptMin", mcmc.acceptance_mean.min()); put("AcceptMax", mcmc.acceptance_mean.max())
put("ChangedMin", 100 * mcmc.changed_fields_mean.min(), 0); put("ChangedMax", 100 * mcmc.changed_fields_mean.max(), 0)
put("FixedStrataMax", 100 * mcmc.fields_in_fixed_strata.max(), 1)
put("RhatMax", mcmc.rhat_max.max(), 3); put("ESSMin", mcmc.ess_min_top60.min(), 0); put("LagOneAbsMax", mcmc.lag1_p90_abs.max())
for macro, key, factor, digits in [("CppCorr", "corr_expected", 1, 6), ("CppZsd", "sd_z", 1, 2),
                                    ("CppZmean", "mean_z", 1, 2), ("CppShareZ", "share_abs_z_gt_1.96", 100, 1)]:
    put(macro, factor * xc.get(key, np.nan), digits)
put("UniformPMin", uni.chi2_p_uniform.min()); put("UniformStatesMax", int(uni.valid_states.max()), 0)
for s, nm in [("N2_clean", "Clean"), ("N2_allmedia", "Allmedia"), ("N1_k2", "Ktwo")]:
    put(f"NActions{nm}", int(mcmc.loc[s, "actions"]), 0); put(f"NMulti{nm}", int(mcmc.loc[s, "multi_tactic_actions"]), 0)
new_values = list(macros.items())[section_start:]
display(pd.DataFrame(new_values, columns=["Macro", "Value"]))

,Macro,Value
0,NStrataNtwo,"1{,}536"
1,NStrataNone,204
2,NFields,"47{,}097"
3,AcceptMin,0.38
4,AcceptMax,0.62
5,ChangedMin,69
6,ChangedMax,87
7,FixedStrataMax,1.2
8,RhatMax,1.013
9,ESSMin,800


## 3. Full-period structure

In [4]:
section_start = len(macros)
glob = R("global_structure_tests").set_index("reference"); elig = R("eligibility_primary"); agree = R("reference_agreement_full_period").set_index("reference")
put("NEligible", int(elig.eligible.sum()), 0)
for s, nm in [("N2", "Ntwo"), ("N1", "None"), ("N2_clean", "Clean"), ("N2_allmedia", "Allmedia"), ("N2_order", "Order"), ("N1_k2", "Ktwo")]:
    g = glob.loc[s]
    put(f"NEval{nm}", int(g.evaluable_pairs), 0); put(f"GlobalRatio{nm}", g.excess_ratio, 0 if g.excess_ratio >= 10 else 1)
    put(f"NExcessQ{nm}", int(g.excess_q05), 0); put(f"NDeficitQ{nm}", int(g.deficit_q05), 0)
    put(f"NStrongExcess{nm}", int(g.strong_excess), 0); put(f"NStrongDeficit{nm}", int(g.strong_deficit), 0)
    put(f"NNeverObs{nm}", int(g.never_observed), 0); put(f"GlobalD{nm}", g.global_discrepancy, 0); put(f"GlobalRef{nm}", g.reference_mean, 0)
put("GlobalPmin", glob.p_mc.max(), 3)
put("CorrNoneNtwo", agree.loc["N2", "pearson_vs_N1"]); put("CorrNoneKtwo", agree.loc["N1_k2", "pearson_vs_N1"])
fam = R("family_structure"); f2 = fam[fam.reference == "N2"]
short = {"A Peaceful I": "A", "B Peaceful II": "B", "C Artistic": "C", "D Disruptive": "D", "E Self-destructive": "E", "F Violent": "F"}
for _, r in f2.iterrows():
    a, b = sorted([short[r.family_a], short[r.family_b]])
    put(f"Fam{a}{b}", r.log2_ratio); put(f"FamRatio{a}{b}", 2 ** r.log2_ratio)
new_values = list(macros.items())[section_start:]
display(pd.DataFrame(new_values, columns=["Macro", "Value"]))

,Macro,Value
0,NEligible,291
1,NEvalNtwo,274
2,GlobalRatioNtwo,50
3,NExcessQNtwo,57
4,NDeficitQNtwo,119
...,...,...
95,FamRatioEE,4.55
96,FamEF,$-$2.55
97,FamRatioEF,0.17
98,FamFF,0.47


## 4. Year-by-year stability

In [5]:
section_start = len(macros)
yvr = R("annual_year_vs_rest"); cal = R("annual_calibration"); lag = R("annual_lag_profile"); perm = R("permutation_mean_year_vs_rest").set_index("reference")
pil = R("pilot_windows_reproduced").set_index("reference"); adj = R("annual_adjacent_years"); sens = R("annual_sensitivity_variants")
for s, nm in [("N2", "Ntwo"), ("N1", "None"), ("N2_clean", "Clean"), ("N2_allmedia", "Allmedia"), ("N2_order", "Order"), ("N1_k2", "Ktwo")]:
    y = yvr[yvr.reference == s].set_index("year")
    put(f"YvrMin{nm}", y.pearson.min()); put(f"YvrMinYear{nm}", str(int(y.pearson.idxmin())))
    put(f"YvrMax{nm}", y.pearson.max()); put(f"YvrMaxYear{nm}", str(int(y.pearson.idxmax()))); put(f"YvrMed{nm}", y.pearson.median())
    put(f"YvrPairsMin{nm}", int(y.pairs.min()), 0); put(f"YvrPairsMax{nm}", int(y.pairs.max()), 0)
    put(f"StrongSignMin{nm}", 100 * y.strong_sign_agreement.min(), 0); put(f"StrongSignMed{nm}", 100 * y.strong_sign_agreement.median(), 0)
    put(f"Pilot{nm}", pil.loc[s, "pearson"], 3); put(f"PilotPairs{nm}", int(pil.loc[s, "pairs"]), 0)
    if not np.isnan(pil.loc[s, "pilot_published_pearson"]):
        put(f"PilotPub{nm}", pil.loc[s, "pilot_published_pearson"], 3)
    a = adj[adj.reference == s]; put(f"AdjMed{nm}", a.pearson.median()); put(f"AdjMin{nm}", a.pearson.min())
    l = lag[lag.reference == s].set_index("lag_years"); put(f"LagOne{nm}", l.loc[1, "mean_pearson"]); put(f"LagSixteen{nm}", l.loc[16, "mean_pearson"])
    put(f"LagMin{nm}", l.mean_pearson.min()); put(f"LagMax{nm}", l.mean_pearson.max())
    if s in cal.reference.unique():
        c = cal[cal.reference == s].set_index("year")
        put(f"CeilMin{nm}", c.ceiling.min()); put(f"CeilMax{nm}", c.ceiling.max())
        put(f"DisMed{nm}", c.disattenuated.median()); put(f"DisMin{nm}", c.disattenuated.min()); put(f"DisMinYear{nm}", str(int(c.disattenuated.idxmin())))
        put(f"DisMax{nm}", c.disattenuated.max())
        for yr in (2019, 2020):
            put(f"Yvr{nm}Y{'Nineteen' if yr == 2019 else 'Twenty'}", y.loc[yr, "pearson"]); put(f"Dis{nm}Y{'Nineteen' if yr == 2019 else 'Twenty'}", c.loc[yr, "disattenuated"])
    if s in perm.index:
        put(f"PermObs{nm}", perm.loc[s, "observed_mean_r"], 3); put(f"PermMean{nm}", perm.loc[s, "perm_mean"], 3); put(f"PermP{nm}", perm.loc[s, "p_lower"], 3)
cs = sens[(sens.reference == "N2") & (sens.variant == "common_support")]; put("CommonPairsNtwo", int(cs.pairs.iloc[0]), 0)
put("CommonMinNtwo", cs.pearson.min()); put("CommonMaxNtwo", cs.pearson.max())
eq = sens[(sens.reference == "N2") & (sens.variant == "equal_year_weight")]; put("EqMinNtwo", eq.pearson.min()); put("EqMaxNtwo", eq.pearson.max())
pf = sens[(sens.reference == "N2") & (sens.variant == "pilot_frozen_selection")]; put("PfMinNtwo", pf.pearson.min()); put("PfMaxNtwo", pf.pearson.max())
mc = R("monte_carlo_error_year_vs_rest"); put("MCRangeMax", mc.pearson_chain_range.max(), 3)
for s, nm in [("N2", "Ntwo"), ("N1", "None")]:
    m_ = mc[mc.reference == s].set_index("year").pearson_chain_range
    put(f"MCMed{nm}", m_.median(), 3); put(f"MCMax{nm}", m_.max(), 2); put(f"MCMaxYear{nm}", str(int(m_.idxmax())))
cb = json.loads((ROOT / "results" / "01_codebook_check.json").read_text())
put("CodebookIdentical", {True: "identical", False: "different", None: "not assessed"}[cb.get("p29_annex_identical")])
new_values = list(macros.items())[section_start:]
display(pd.DataFrame(new_values, columns=["Macro", "Value"]))

,Macro,Value
0,YvrMinNtwo,0.70
1,YvrMinYearNtwo,2020
2,YvrMaxNtwo,0.92
3,YvrMaxYearNtwo,2011
4,YvrMedNtwo,0.83
...,...,...
156,MCMaxYearNtwo,2024
157,MCMedNone,0.005
158,MCMaxNone,0.02
159,MCMaxYearNone,2008


## 5. Log-linear models and calibrated departures

In [6]:
section_start = len(macros)
dec = R("loglinear_decomposition").set_index("reference"); Fp = R("departure_F_pooled").set_index("reference"); Ft = R("annual_departure_F")
phi = R("unidiff_phi_by_year"); checks = json.loads((ROOT / "results" / "05_fit_checks.json").read_text())
put("GLMDiff", checks["max_abs_fitted_diff"] * 1e10, 1); put("PhiDiff", checks["max_abs_phi_diff"] * 1e4, 1)
for s, nm in [("N2", "Ntwo"), ("N1", "None"), ("N2_clean", "Clean")]:
    d = dec.loc[s]
    put(f"FreqRaw{nm}", 100 * d.frequency_share_raw, 0); put(f"FreqDisp{nm}", 100 * d.frequency_share_dispersion_floor, 0)
    put(f"FreqPerm{nm}", 100 * d.frequency_share_permutation_floor, 0) if np.isfinite(d.frequency_share_permutation_floor) else None
    put(f"StructConst{nm}", 100 * d.share_structure_constant, 0); put(f"Uniform{nm}", 100 * d.share_change_uniform_strength, 1)
    put(f"FUniform{nm}", d.F_uniform_strength); put(f"PUniform{nm}", d.p_F_uniform_strength, 3)
    put(f"DMzero{nm}", d.D_M0, 0); put(f"DMone{nm}", d.D_M1, 0); put(f"DMtwo{nm}", d.D_M2, 0); put(f"DMA{nm}", d.D_MA, 0)
    put(f"PermDMone{nm}", d.perm_mean_D_M1, 0); put(f"Disp{nm}", d.pooled_dispersion_within)
    f = Fp.loc[s]; put(f"FPooled{nm}", f.F_pooled); put(f"BeyondNoise{nm}", 100 * f.share_beyond_noise_pooled, 0)
    put(f"NYearsDepart{nm}", int(f.years_p_holm_lt_05), 0)
    yrs = str(f.years_list) if isinstance(f.years_list, str) else ""
    put(f"YearsDepart{nm}", yrs.replace(",", ", ") if yrs else "none")
    ft = Ft[Ft.reference == s].set_index("year")
    for yr, w in [(2019, "Nineteen"), (2020, "Twenty"), (2016, "Sixteen"), (2021, "TwentyOne")]:
        put(f"F{nm}Y{w}", ft.loc[yr, "F"]); put(f"Beyond{nm}Y{w}", 100 * ft.loc[yr, "share_beyond_noise"], 0)
    others = ft[ft.p_holm >= 0.05]; put(f"FOtherMin{nm}", others.F.min()); put(f"FOtherMax{nm}", others.F.max())
    p = phi[phi.reference == s].set_index("year")
    put(f"PhiMin{nm}", p.phi.min()); put(f"PhiMinYear{nm}", str(int(p.phi.idxmin()))); put(f"PhiMax{nm}", p.phi.max()); put(f"PhiMaxYear{nm}", str(int(p.phi.idxmax())))
    for yr, w in [(2019, "Nineteen"), (2020, "Twenty"), (2021, "TwentyOne")]:
        put(f"Phi{nm}Y{w}", p.loc[yr, "phi"]); put(f"Phi{nm}Y{w}Lo", p.loc[yr, "phi_lo"]); put(f"Phi{nm}Y{w}Hi", p.loc[yr, "phi_hi"])
    put(f"NPhiBelow{nm}", int((p.phi_hi < 1).sum()), 0); put(f"NPhiAbove{nm}", int((p.phi_lo > 1).sum()), 0)
new_values = list(macros.items())[section_start:]
display(pd.DataFrame(new_values, columns=["Macro", "Value"]))

,Macro,Value
0,GLMDiff,1.9
1,PhiDiff,2.0
2,FreqRawNtwo,71
3,FreqDispNtwo,96
4,FreqPermNtwo,117
...,...,...
123,PhiCleanYTwentyOne,0.87
124,PhiCleanYTwentyOneLo,0.71
125,PhiCleanYTwentyOneHi,1.12
126,NPhiBelowClean,0


## 5b. Change in tactic use versus pairing, and family ratios by year (Figs. 1 and 3a)

In [7]:
section_start = len(macros)
chg = R("change_summary").set_index("reference"); cby = R("change_by_year"); fyr = R("family_ratio_by_year")
put("ChgTacN", int(chg.loc["N2", "tactics"]), 0); put("ChgTacBeyond", int(chg.loc["N2", "tactics_beyond_noise"]), 0)
put("ChgTacMedFold", 2 ** chg.loc["N2", "tactics_median_abs_change"], 1)
for s, nm in [("N2", "Ntwo"), ("N1", "None")]:
    c = chg.loc[s]
    put(f"ChgPairs{nm}", int(c.pairs), 0); put(f"ChgShareMedFold{nm}", 2 ** c.pairs_median_abs_share_change, 1)
    put(f"ChgAssocMedFold{nm}", 2 ** c.pairs_median_abs_association_change, 1); put(f"ChgAssocBeyond{nm}", int(c.pairs_association_beyond_noise), 0)
    put(f"ChgR{nm}", c.r_expected_observed); put(f"ChgRed{nm}", 100 * c.reduction_vs_unchanged_share, 0)
    put(f"ChgBig{nm}", int(c.pairs_doubled_or_halved), 0); put(f"ChgBigExp{nm}", int(c.doubled_or_halved_in_expected_direction), 0)
    y = cby[cby.reference == s].set_index("year")
    put(f"ChgYearRMed{nm}", y.r_expected_observed.median()); put(f"ChgYearRMin{nm}", y.r_expected_observed.min())
    put(f"ChgYearRMax{nm}", y.r_expected_observed.max()); put(f"ChgYearRMinYear{nm}", str(int(y.r_expected_observed.idxmin())))
    put(f"ChgYearRedMed{nm}", 100 * y.reduction_vs_unchanged_share.median(), 0)
put("ChgYearRNineteenNtwo", cby[(cby.reference == "N2") & (cby.year == 2019)].r_expected_observed.iloc[0])
f2y = fyr[fyr.reference == "N2"]
HEADFAM = {("C Artistic", "D Disruptive"): "ArtDis", ("C Artistic", "F Violent"): "ArtVio", ("B Peaceful II", "D Disruptive"): "PeaceTwoDis",
           ("A Peaceful I", "B Peaceful II"): "PeaceOnePeaceTwo", ("A Peaceful I", "C Artistic"): "PeaceOneArt", ("A Peaceful I", "D Disruptive"): "PeaceOneDis"}
for (fa, fb), nm in HEADFAM.items():
    z = f2y[(f2y.family_a == fa) & (f2y.family_b == fb)].set_index("year")
    put(f"Fam{nm}Min", z.ratio.min()); put(f"Fam{nm}Max", z.ratio.max())
    put(f"Fam{nm}Below", int((z.ratio < 1).sum()), 0); put(f"Fam{nm}Above", int((z.ratio > 1).sum()), 0)
    put(f"Fam{nm}CIBelow", int((z.hi < 1).sum()), 0); put(f"Fam{nm}CIAbove", int((z.lo > 1).sum()), 0)
    put(f"Fam{nm}TopThree", ", ".join(str(v) for v in sorted(z.ratio.nlargest(3).index)))
pd.DataFrame(list(macros.items())[section_start:], columns=["Macro", "Value"])

,Macro,Value
0,ChgTacN,32
1,ChgTacBeyond,21
2,ChgTacMedFold,1.8
3,ChgPairsNtwo,108
4,ChgShareMedFoldNtwo,1.8
...,...,...
67,FamPeaceOneDisBelow,11
68,FamPeaceOneDisAbove,6
69,FamPeaceOneDisCIBelow,1
70,FamPeaceOneDisCIAbove,1


## 6. Write `numbers.tex`

In [8]:
macro_table = pd.DataFrame(sorted(macros.items()), columns=["Macro", "Value"])
display(macro_table)
lines = ["% Generated by notebooks/08_manuscript_exports.ipynb from results/tables. Do not edit by hand."]
lines.extend(f"\\newcommand{{\\{key}}}{{{value}}}" for key, value in macro_table.itertuples(index=False))
(OUT / "numbers.tex").write_text("\n".join(lines) + "\n")
print(f"Exported {len(macro_table)} numerical cross-checks.")

,Macro,Value
0,AcceptMax,0.62
1,AcceptMin,0.38
2,ActionsMax,"3{,}374"
3,ActionsMaxYear,2019
4,ActionsMin,732
...,...,...
562,YvrPairsMinClean,9
563,YvrPairsMinKtwo,10
564,YvrPairsMinNone,28
565,YvrPairsMinNtwo,28


Exported 567 numerical cross-checks.


## 7. LaTeX tables (main text and Supporting Information)

In [9]:
def tex_table(df, path, colfmt=None, longtable=False):
    display(df)
    # a "\n" inside a column name stacks that header over several lines
    heads = [h if "\n" not in h else "\\begin{tabular}[b]{@{}c@{}}" + h.replace("\n", "\\\\") + "\\end{tabular}" for h in df.columns]
    body = df.set_axis(heads, axis=1).to_latex(index=False, escape=False, column_format=colfmt, longtable=longtable, na_rep="--")
    body = body.replace("NaN", "--")
    (OUT / "tables" / path).write_text(body)
def f2(x, d=2):
    return "--" if pd.isna(x) else (f"{x:.{d}f}".replace("-", "$-$") if x < 0 else f"{x:.{d}f}")
LAB = {"N2": "N2 context and field", "N1": "N1 month", "N2_clean": "N2 without generic codes", "N2_allmedia": "N2 all media",
       "N2_order": "N2 order preserved", "N1_k2": "N1 two-tactic actions"}
SHORT = {"N2": "N2", "N1": "N1", "N2_clean": "N2 clean", "N2_allmedia": "N2 all media", "N2_order": "N2 order", "N1_k2": "N1 two tactics"}
RAG = ">{\\raggedright\\arraybackslash}"
refs = ["N2", "N1", "N2_clean", "N2_allmedia", "N2_order", "N1_k2"]
# Table 1: one robustness table, six references (sampler diagnostics are in the Supporting table on the samplers)
def year_ranges(s):
    if not isinstance(s, str) or not s:
        return "none"
    ys = sorted(int(v) for v in s.split(",")); out, a = [], ys[0]
    for p, q in zip(ys, ys[1:] + [None]):
        if q != p + 1:
            out.append(str(a) if a == p else f"{a}--{p}"); a = q
    return ", ".join(out)
rows = []
for s in refs:
    y = yvr[yvr.reference == s]; l = lag[lag.reference == s].set_index("lag_years"); c = cal[cal.reference == s]
    rows.append({"Evaluable pairs": str(int(glob.loc[s, "evaluable_pairs"])),
                 "Overall departure": f"{glob.loc[s, 'excess_ratio']:.0f}",
                 "Excess / deficit pairs": f"{int(glob.loc[s, 'excess_q05'])} / {int(glob.loc[s, 'deficit_q05'])}",
                 "Median $r$ with other years": f2(y.pearson.median()),
                 "\\quad range": f"{f2(y.pearson.min())}--{f2(y.pearson.max())}",
                 "\\quad relative to noise ceiling": f2(c.disattenuated.median()) if len(c) else "--",
                 "$r$ at 1 / 16 years apart": f"{f2(l.loc[1, 'mean_pearson'])} / {f2(l.loc[16, 'mean_pearson'])}",
                 "Years beyond noise": (", ".join("\\mbox{" + v + "}" for v in year_ranges(Fp.loc[s, "years_list"]).split(", "))
                                        if s in Fp.index else "--"),
                 "Fit gain (\\%): deviance": f"{100 * dec.loc[s, 'frequency_share_raw']:.0f}" if s in dec.index else "--",
                 "\\quad adjusted discrepancy": f"{100 * dec.loc[s, 'frequency_share_dispersion_floor']:.0f}" if s in dec.index else "--"})
# Measures in rows and the six references in columns, so each row reads across references.
t1 = pd.DataFrame(rows, index=[LAB[s] for s in refs]).T
display(t1)
GROUPS = [("Full period", ["Evaluable pairs", "Overall departure", "Excess / deficit pairs"]),
          ("Year to year", ["Median $r$ with other years", "\\quad range", "\\quad relative to noise ceiling", "$r$ at 1 / 16 years apart"]),
          ("Departures and fit", ["Years beyond noise", "Fit gain (\\%): deviance", "\\quad adjusted discrepancy"])]
assert [m for _, g in GROUPS for m in g] == list(t1.index)
HEAD = {"N2": "N2\\\\context\\\\and field", "N1": "N1\\\\month", "N2_clean": "N2\\\\without\\\\generic\\\\codes",
        "N2_allmedia": "N2\\\\all\\\\media", "N2_order": "N2\\\\order\\\\preserved", "N1_k2": "N1\\\\two-tactic\\\\actions"}
CEN = ">{\\centering\\arraybackslash}"
lines = ["\\begin{tabular}{" + RAG + "p{4.25cm}" + (CEN + "p{1.7cm}") * len(refs) + "}", "\\toprule",
         " & " + " & ".join("\\begin{tabular}[b]{@{}c@{}}" + HEAD[s] + "\\end{tabular}" for s in refs) + " \\\\", "\\midrule"]
for gi, (g, items) in enumerate(GROUPS):
    if gi:
        lines.append("\\addlinespace")
    lines.append(f"\\multicolumn{{{len(refs) + 1}}}{{@{{}}l}}{{\\textit{{{g}}}}} \\\\")
    lines += [m + " & " + " & ".join(t1.loc[m, LAB[s]] for s in refs) + " \\\\" for m in items]
lines += ["\\bottomrule", "\\end{tabular}"]
(OUT / "tables" / "table1_robustness.tex").write_text("\n".join(lines) + "\n")


,N2 context and field,N1 month,N2 without generic codes,N2 all media,N2 order preserved,N1 two-tactic actions
Evaluable pairs,274,291,209,280,272,142
Overall departure,50,94,37,56,39,51
Excess / deficit pairs,57 / 119,68 / 134,39 / 97,56 / 125,58 / 117,26 / 58
Median $r$ with other years,0.83,0.85,0.87,0.84,0.82,0.85
\quad range,0.70--0.92,0.75--0.90,0.76--0.96,0.74--0.92,0.66--0.91,0.69--0.92
\quad relative to noise ceiling,0.97,0.93,0.97,--,--,--
$r$ at 1 / 16 years apart,0.77 / 0.79,0.81 / 0.72,0.81 / 0.98,0.78 / 0.81,0.75 / 0.78,0.85 / 0.71
Years beyond noise,"\mbox{2016}, \mbox{2019--2021}","\mbox{2011}, \mbox{2013}, \mbox{2015--2016}, \...",\mbox{2020--2021},--,--,--
Fit gain (\%): deviance,71,62,66,--,--,--
\quad adjusted discrepancy,96,87,89,--,--,--


1689

In [10]:
# Supporting tables (file names describe content; si.tex sets the S numbers)
dic = pd.read_csv(TAB / "tactic_dictionary.csv")
tot = marg.sum()
s1 = dic.assign(Mentions=dic.code.map(lambda c: f"{int(tot[c]):,}"))[["code", "label_en", "label_es", "family_label", "Mentions"]]
s1.columns = ["Code", "Label (English)", "Label in codebook (Spanish)", "Family", "Actions"]
tex_table(s1, "si_tab_dictionary.tex", "rp{4cm}p{5cm}lr")
s2 = fn.reset_index()[["year", "actions", "no_valid_tactic", "repeated_valid_code", "eligible", "eligible_two_plus", "eligible_with_residual_99", "eligible_six_fields"]].copy()
s2.columns = ["Year", "Newspaper\nactions", "No valid\ntactic", "Repeated\ncode", "Eligible", "Two or more\ntactics", "With\nresidual 99", "Six\nfields"]
tex_table(s2, "si_tab_flow.tex", "rrrrrrrr")
s3 = cn.reset_index()[["year", "actions", "multi_tactic", "mean_k", "share_regional_press", "share_national_press", "share_combined", "share_metropolitan", "share_p20_missing"]].copy()
for ccol in ["mean_k", "share_regional_press", "share_national_press", "share_combined", "share_metropolitan", "share_p20_missing"]:
    s3[ccol] = s3[ccol].map(f2)
s3.columns = ["Year", "Actions", "Two or\nmore", "Mean\ntactics", "Regional\npress", "National\npress", "Combined", "Metropolitan\nregion", "P20a\nmissing"]
tex_table(s3, "si_tab_coverage.tex", "rrrrrrrrr")
s4 = mcmc.reset_index()[["spec", "actions", "multi_tactic_actions", "occupied_fields", "strata", "fields_in_fixed_strata", "acceptance_mean", "changed_fields_mean", "rhat_max", "lag1_p90_abs", "ess_min_top60"]].copy()
s4["spec"] = s4.spec.map(LAB)
for ccol in ["fields_in_fixed_strata", "acceptance_mean", "changed_fields_mean", "lag1_p90_abs"]:
    s4[ccol] = s4[ccol].map(f2)
s4["rhat_max"] = s4.rhat_max.map(lambda x: f"{x:.3f}"); s4["ess_min_top60"] = s4.ess_min_top60.map(lambda x: f"{x:.0f}")
s4.columns = ["Reference", "Actions", "Two or\nmore", "Fields", "Strata", "Fixed\nstrata", "Mean\nacceptance", "Fields\nchanged", "Max\n$\\hat{R}$", "p90 of\n$|$lag-1$|$", "Min\nESS"]
tex_table(s4, "si_tab_mcmc.tex", RAG + "p{2.5cm}rrrrrrrrrr")
s5 = fam[fam.reference.isin(["N2", "N1"])].copy()
s5 = s5.assign(reference=s5.reference.map(SHORT), expected=s5.expected.map(lambda x: f"{x:.1f}"), log2_ratio=s5.log2_ratio.map(f2), z=s5.z.map(lambda x: f2(x, 1)))
s5 = s5[["reference", "family_a", "family_b", "pairs", "observed", "expected", "log2_ratio", "z"]]
s5.columns = ["Reference", "Family", "Family", "Pairs", "Observed", "Expected", "log$_2$\nratio", "$z$"]
tex_table(s5, "si_tab_families.tex", "lllrrrrr", longtable=True)
pn = R("pairs_full_period_N2"); pn = pn[pn.evaluable].sort_values("log2_ratio", ascending=False)
s6 = pd.DataFrame({"Codes": pn.code_i.astype(str) + "--" + pn.code_j.astype(str), "Tactics": pn.label_i + " / " + pn.label_j,
                   "Observed": pn.observed, "Expected": pn.expected.map(lambda x: f"{x:.1f}"), "log$_2$ ratio": pn.log2_ratio.map(f2),
                   "$q$": pn.q_bh_595.map(lambda x: "$<$0.001" if x < 0.001 else f"{x:.3f}")})
tex_table(s6, "si_tab_pairs_N2.tex", "lp{7cm}rrrr", longtable=True)
yy = yvr[yvr.reference.isin(["N2", "N1"])].merge(cal[["reference", "year", "ceiling", "disattenuated"]], on=["reference", "year"], how="left")
s7 = pd.DataFrame({"Reference": yy.reference.map(SHORT), "Year": yy.year, "Pairs": yy.pairs, "Pearson": yy.pearson.map(f2), "Spearman": yy.spearman.map(f2),
                   "Strong-sign\nagreement": yy.strong_sign_agreement.map(f2), "Ceiling": yy.ceiling.map(f2), "Disattenuated": yy.disattenuated.map(f2)})
tex_table(s7, "si_tab_year_vs_rest.tex", "lrrrrrrr", longtable=True)
ftab = Ft[Ft.reference.isin(["N2", "N1", "N2_clean"])]
s8 = pd.DataFrame({"Reference": ftab.reference.map(SHORT), "Year": ftab.year, "Cells": ftab.cells, "Within-year\ndispersion": ftab.dispersion_within.map(f2),
                   "$F$": ftab.F.map(f2), "Beyond\nnoise (\\%)": (100 * ftab.share_beyond_noise).map(lambda x: f"{x:.0f}"),
                   "Holm $p$": ftab.p_holm.map(lambda x: "$<$0.001" if x < 0.001 else f"{x:.3f}")})
tex_table(s8, "si_tab_departure_F.tex", "lrrrrrr", longtable=True)
ph = phi.copy()
s9 = pd.DataFrame({"Reference": ph.reference.map(SHORT), "Year": ph.year, "$\\varphi_t$": ph.phi.map(f2), "95\\% interval": ph.phi_lo.map(f2) + "--" + ph.phi_hi.map(f2)})
tex_table(s9, "si_tab_phi.tex", "lrrr", longtable=True)
dd = dec.copy()
c10 = [s for s in refs if s in dd.index]
n0 = lambda x: "--" if pd.isna(x) else f"{x:,.0f}"
q10 = [("Deviance, M0 (year effects only)", "D_M0", n0), ("Deviance, M1 (constant structure)", "D_M1", n0),
       ("Deviance, M2 (uniform strength)", "D_M2", n0), ("Deviance, MA (constant pair shares)", "D_MA", n0),
       ("Permutation floor for D(M1)", "perm_mean_D_M1", n0), ("Within-year dispersion", "pooled_dispersion_within", f2),
       ("Model fit gain, deviance (\\%)", "frequency_share_raw", lambda x: f"{100 * x:.0f}"),
       ("Model fit gain, dispersion-adjusted (\\%)", "frequency_share_dispersion_floor", lambda x: f"{100 * x:.0f}"),
       ("Uniform-strength share (\\%)", "share_change_uniform_strength", lambda x: f"{100 * x:.1f}"),
       ("$F$, uniform strength", "F_uniform_strength", f2)]
HEAD10 = {"N2": "N2\ncontext\nand field", "N1": "N1\nmonth", "N2_clean": "N2\nwithout\ngeneric codes", "N2_allmedia": "N2\nall\nmedia",
          "N2_order": "N2\norder\npreserved", "N1_k2": "N1\ntwo-tactic\nactions"}
s10 = pd.DataFrame([[lab] + [fmt(dd.loc[s, col]) for s in c10] for lab, col, fmt in q10], columns=["Quantity"] + [HEAD10[s] for s in c10])
tex_table(s10, "si_tab_decomposition.tex", RAG + "p{4.6cm}" + "r" * len(c10))
pr = pil.reset_index()
s11 = pd.DataFrame({"Reference": pr.reference.map(LAB), "Pairs": pr.pairs, "Pearson": pr.pearson.map(lambda x: f"{x:.3f}"),
                    "Spearman": pr.spearman.map(lambda x: f"{x:.3f}"), "Strong-sign agreement": pr.strong_sign_agreement.map(lambda x: f"{x:.3f}")})
tex_table(s11, "si_tab_two_blocks.tex", "lrrrr")
dep = R("departure_pairs_flagged_years_N2")
s12 = pd.DataFrame({"Year": dep.year, "Direction": dep.direction.map({"more_than_common_structure": "more", "fewer_than_common_structure": "fewer"}),
                    "Tactics": dep.label_i + " / " + dep.label_j, "Observed": dep.observed, "Predicted": dep.predicted_common_structure.map(lambda x: f"{x:.1f}"),
                    "Residual": dep.pearson_residual.map(lambda x: f2(x, 1))})
tex_table(s12, "si_tab_departure_pairs.tex", "rlp{7cm}rrr", longtable=True)
sv = sens[sens.reference == "N2"].pivot(index="year", columns="variant", values="pearson").reset_index()
sv = sv.assign(**{c_: sv[c_].map(f2) for c_ in ["common_support", "equal_year_weight", "pilot_frozen_selection"]})
sv.columns = ["Year", "Common support", "Equal year weights", "Pairs selected on 2008--2018"]
tex_table(sv, "si_tab_sensitivity.tex", "rrrr")

,Code,Label (English),Label in codebook (Spanish),Family,Actions
0,1,March,Marcha,A Peaceful I,"6,317"
1,2,Static demonstration,Manifestación,A Peaceful I,"6,597"
2,3,Rally with speakers,Acto con oradores,A Peaceful I,674
3,4,Caravan,Caravana,A Peaceful I,521
4,5,Petition or fund collection,Recolección de firmas y/o dinero,A Peaceful I,720
5,6,Peaceful civil disobedience,Desobediencia civil pacífica,A Peaceful I,43
6,7,Press conference or statement,Conferencia de prensa o declaración pública,B Peaceful II,50
7,8,Assembly or open debate,"Asamblea, discusión o debate abierto",B Peaceful II,50
8,9,Commemoration,Rememoración u homenaje,B Peaceful II,634
9,10,Chants or slogans,"Coros, cánticos, slogans o consignas",B Peaceful II,"6,708"


,Year,Newspaper\nactions,No valid\ntactic,Repeated\ncode,Eligible,Two or more\ntactics,With\nresidual 99,Six\nfields
0,2008,2023,8,0,2015,798,11,3
1,2009,1421,2,0,1419,603,10,1
2,2010,1152,0,0,1152,535,5,2
3,2011,2247,0,1,2246,982,15,14
4,2012,1467,0,0,1467,639,0,6
5,2013,1832,0,0,1832,844,0,10
6,2014,1795,0,0,1795,746,0,6
7,2015,1264,0,0,1264,435,0,1
8,2016,1587,0,0,1587,847,0,3
9,2017,1078,1,0,1077,548,1,0


,Year,Actions,Two or\nmore,Mean\ntactics,Regional\npress,National\npress,Combined,Metropolitan\nregion,P20a\nmissing
0,2008,2015,798,1.57,0.67,0.16,0.17,0.16,0.02
1,2009,1419,603,1.59,0.75,0.12,0.13,0.14,0.02
2,2010,1152,535,1.60,0.74,0.11,0.15,0.16,0.03
3,2011,2246,982,1.67,0.72,0.13,0.15,0.16,0.03
4,2012,1467,639,1.61,0.62,0.15,0.23,0.17,0.05
5,2013,1832,844,1.64,0.69,0.14,0.18,0.14,0.04
6,2014,1795,746,1.55,0.76,0.12,0.12,0.11,0.05
7,2015,1264,435,1.48,0.59,0.22,0.19,0.15,0.05
8,2016,1587,847,1.75,0.59,0.15,0.26,0.14,0.04
9,2017,1077,548,1.68,0.51,0.14,0.35,0.21,0.03


,Reference,Actions,Two or\nmore,Fields,Strata,Fixed\nstrata,Mean\nacceptance,Fields\nchanged,Max\n$\hat{R}$,p90 of\n$|$lag-1$|$,Min\nESS
0,N1 month,27488,13405,47097,204,0.00,0.62,0.87,1.013,0.10,875
1,N2 context and field,27488,13405,47097,1536,0.01,0.58,0.74,1.012,0.10,800
2,N2 without generic codes,23559,6666,32707,1362,0.01,0.61,0.73,1.009,0.11,865
3,N2 all media,34118,15863,56964,1575,0.01,0.60,0.75,1.008,0.10,996
4,N2 order preserved,27488,13405,47097,1536,0.01,0.38,0.69,1.013,0.10,886
5,N1 two-tactic actions,8981,8981,17962,204,0.00,0.55,0.84,1.007,0.10,900


,Reference,Family,Family,Pairs,Observed,Expected,log$_2$\nratio,$z$
0,N1,A Peaceful I,A Peaceful I,15,1691,1851.6,$-$0.13,$-$4.7
1,N1,A Peaceful I,B Peaceful II,48,6697,3639.9,0.88,65.2
2,N1,A Peaceful I,C Artistic,18,1443,847.6,0.77,23.8
3,N1,A Peaceful I,D Disruptive,42,3562,5672.3,$-$0.67,$-$39.6
4,N1,A Peaceful I,E Self-destructive,12,75,212.6,$-$1.50,$-$10.5
5,N1,A Peaceful I,F Violent,54,2657,3211.5,$-$0.27,$-$12.4
6,N1,B Peaceful II,B Peaceful II,28,428,308.3,0.47,8.3
7,N1,B Peaceful II,C Artistic,24,677,419.2,0.69,14.8
8,N1,B Peaceful II,D Disruptive,56,1892,2998.4,$-$0.66,$-$26.1
9,N1,B Peaceful II,E Self-destructive,16,53,111.3,$-$1.06,$-$6.3


,Codes,Tactics,Observed,Expected,log$_2$ ratio,$q$
79,3--16,Rally with speakers / Artistic performance (ar...,85,12.8,2.68,0.008
72,3--9,Rally with speakers / Commemoration,81,20.1,1.98,0.008
487,20--33,Land occupation / Attack on police or security...,43,13.5,1.64,0.008
247,9--13,Commemoration / Vigil,38,11.9,1.63,0.008
250,9--16,Commemoration / Artistic performance (artists),47,15.2,1.60,0.008
...,...,...,...,...,...,...
415,16--27,Artistic performance (artists) / Arson,0,14.4,$-$4.90,0.008
407,16--19,Artistic performance (artists) / Building occu...,0,16.0,$-$5.05,0.008
82,3--19,Rally with speakers / Building occupation,1,52.5,$-$5.14,0.008
113,4--19,Caravan / Building occupation,0,17.4,$-$5.17,0.008


... (3 outputs left out)

,Quantity,N2\ncontext\nand field,N1\nmonth,N2\nwithout\ngeneric codes
0,"Deviance, M0 (year effects only)","14,025","24,068","8,806"
1,"Deviance, M1 (constant structure)","4,350","5,814","2,805"
2,"Deviance, M2 (uniform strength)","4,303","5,511","2,772"
3,"Deviance, MA (constant pair shares)","15,127","15,127","8,137"
4,Permutation floor for D(M1),"5,477","4,853","3,550"
5,Within-year dispersion,1.30,1.35,1.28
6,"Model fit gain, deviance (\%)",71,62,66
7,"Model fit gain, dispersion-adjusted (\%)",96,87,89
8,Uniform-strength share (\%),1.1,5.2,1.2
9,"$F$, uniform strength",2.26,14.00,1.58


,Reference,Pairs,Pearson,Spearman,Strong-sign agreement
0,N1 month,113,0.858,0.891,0.976
1,N2 context and field,108,0.829,0.843,0.953
2,N2 without generic codes,65,0.801,0.845,0.957
3,N2 all media,117,0.839,0.865,0.947
4,N2 order preserved,104,0.813,0.816,0.967
5,N1 two-tactic actions,41,0.855,0.852,0.967


,Year,Direction,Tactics,Observed,Predicted,Residual
0,2016,more,Static demonstration / Building occupation,30,12.1,5.1
1,2016,more,Press conference or statement / Chants or slogans,11,3.9,3.6
2,2016,more,Static demonstration / Roadblock or barricade,94,65.8,3.5
3,2016,more,March / Boycott,7,2.1,3.4
4,2016,more,Strike or stoppage / Arson,1,0.1,3.4
...,...,...,...,...,...,...
59,2021,fewer,Static demonstration / Chants or slogans,371,411.6,$-$2.0
60,2021,fewer,Roadblock or barricade / Attack on police or s...,86,106.5,$-$2.0
61,2021,fewer,Static demonstration / Attack on bystanders,1,5.7,$-$2.0
62,2021,fewer,Arson / Looting,2,6.4,$-$1.7


,Year,Common support,Equal year weights,Pairs selected on 2008--2018
0,2008,0.88,0.79,0.80
1,2009,0.87,0.84,0.83
2,2010,0.95,0.89,0.88
3,2011,0.96,0.91,0.92
4,2012,0.81,0.86,0.86
5,2013,0.86,0.90,0.89
6,2014,0.85,0.88,0.88
7,2015,0.91,0.79,0.77
8,2016,0.92,0.74,0.71
9,2017,0.90,0.88,0.88
